In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG  —  EMNIST ByClass: 814,255 chars, 62 unbalanced classes (0-9, A-Z, a-z)
# ─────────────────────────────────────────────────────────────────────────────
DATASET      = "emnist/byclass"
NUM_CLASSES  = 62          # 0-9 (10) + A-Z (26) + a-z (26)
IMG          = 28
BS           = 64
EPOCHS       = 60
LR           = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05        # slight smoothing handles class imbalance noise
DROP_PATH    = 0.10
WARMUP_EP    = 5
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/byclass"
AUTOTUNE     = tf.data.AUTOTUNE

# label mapping (already 0-indexed by tfds): 0-9 digits, 10-35 A-Z, 36-61 a-z
LABELS = (
    [str(d) for d in range(10)]
    + [chr(c) for c in range(65, 91)]
    + [chr(c) for c in range(97, 123)]
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
print(f"[INFO] Loading {DATASET} via tensorflow_datasets ...")
train_raw, info = tfds.load(
    DATASET, split="train", as_supervised=True, with_info=True, shuffle_files=True,
)
test_raw = tfds.load(DATASET, split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    label = tf.cast(label, tf.int32)
    return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.20)
    image = tf.image.random_contrast(image, 0.80, 1.20)
    image = tf.pad(image, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    image = tf.image.random_flip_left_right(image)
    image = image + tf.random.normal(tf.shape(image), stddev=0.02)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

train_raw = train_raw.shuffle(20_000, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds    = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BS).prefetch(AUTOTUNE)
test_ds_oh = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    return tf.nn.gelu(x)

class DropPath(keras.layers.Layer):
    def __init__(self, drop_prob=0.0, **kwargs):
        super().__init__(**kwargs)
        self.drop_prob = drop_prob
    def call(self, x, training=None):
        if not training or self.drop_prob == 0.0:
            return x
        keep  = 1.0 - self.drop_prob
        shape = (tf.shape(x)[0],) + (1,) * (len(x.shape) - 1)
        mask  = tf.floor(keep + tf.random.uniform(shape, dtype=x.dtype))
        return x * mask / keep
    def get_config(self):
        cfg = super().get_config(); cfg["drop_prob"] = self.drop_prob; return cfg

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels, drop_rate=0.0):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if drop_rate > 0:
        x = DropPath(drop_rate)(x)
    x = layers.Add()([x, r])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch, drop_rate=0.0):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch, drop_rate)
    r2  = residual_block(r1, out_ch, drop_rate)
    r3  = residual_block(r2, out_ch, drop_rate)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def english_topology_module(x, out_features):
    """Symmetric kernels + dilated convs for Latin strokes (curves, ascenders, descenders)."""
    h  = layers.Conv2D(48, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    v  = layers.Conv2D(48, (5, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    d4 = layers.Conv2D(32, 3, padding="same", dilation_rate=4, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2, d4])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    return layers.Dense(out_features, activation=gelu)(out)

# ─────────────────────────────────────────────────────────────────────────────
#  MODEL
# ─────────────────────────────────────────────────────────────────────────────
def build_model(num_classes=62, image_size=28, drop_path_rate=0.10):
    K, DR = num_classes, drop_path_rate
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # Stem: 3-path for curves, horizontals, verticals
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])          # 64ch
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    enc1 = dense_res_block(stem, 64,  64,  drop_rate=DR * 0.5)
    enc2 = dense_res_block(enc1, 64,  128, drop_rate=DR)
    enc3 = dense_res_block(enc2, 128, 256, drop_rate=DR)

    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.GlobalAveragePooling2D()(enc3)
    multi_scale = layers.Concatenate()([gap1, gap2, gap3])  # 512-dim

    stm_out = english_topology_module(enc3, 256)

    fused = layers.Concatenate()([stm_out, multi_scale])    # 768-dim
    x   = layers.Dense(256, use_bias=False)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    x   = layers.Dropout(0.25)(x)
    out = layers.Dense(K, name="logits")(x)
    return Model(inputs=inp, outputs=out, name="WhatNet_ByClass")

# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────────────────────
model        = build_model(NUM_CLASSES, IMG, DROP_PATH)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=12,
                      restore_best_weights=True, verbose=1),
    ],
)

# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

# ─────────────────────────────────────────────────────────────────────────────
#  SAVE
# ─────────────────────────────────────────────────────────────────────────────
results = {
    "dataset": "EMNIST_BYCLASS", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)
print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

# Method

In [ ]:
import os, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG  —  EMNIST ByClass: 814,255 chars, 62 UNBALANCED classes
#  label mapping (0-indexed): 0-9 digits, 10-35 A-Z, 36-61 a-z
#  Hardest EMNIST split: uppercase and lowercase are SEPARATE classes,
#  so 'A'≠'a'. Key confusions: c/C, o/O, p/P, u/U, v/V, w/W, x/X.
#  Severe imbalance — macro F1 is the primary metric, not accuracy.
# ─────────────────────────────────────────────────────────────────────────────
DATASET      = "emnist/byclass"
NUM_CLASSES  = 62
IMG          = 28
BS           = 64
EPOCHS       = 1
LR           = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.10          # severe imbalance + case separation → highest smoothing
WARMUP_EP    = 5
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/byclass"
AUTOTUNE     = tf.data.AUTOTUNE

LABELS = (
    [str(d) for d in range(10)]
    + [chr(c) for c in range(65, 91)]   # A-Z
    + [chr(c) for c in range(97, 123)]  # a-z
)  # 62 labels
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
print("[INFO] Loading EMNIST ByClass via tensorflow_datasets ...")
train_raw, info = tfds.load(
    DATASET, split="train", as_supervised=True, with_info=True, shuffle_files=True,
)
test_raw = tfds.load(DATASET, split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

# ─────────────────────────────────────────────────────────────────────────────
#  CLASS WEIGHTS  —  scan labels to counter severe imbalance
# ─────────────────────────────────────────────────────────────────────────────
print("[INFO] Computing class weights from training labels ...")
label_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, label in train_raw.take(n_train):
    label_counts[int(label.numpy())] += 1

total_samples = label_counts.sum()
class_weights = total_samples / (NUM_CLASSES * np.maximum(label_counts, 1))
class_weights = np.clip(class_weights, 0.1, 10.0)
class_weight_dict = {i: float(class_weights[i]) for i in range(NUM_CLASSES)}
print(f"[INFO] Class weight range: [{class_weights.min():.2f}, {class_weights.max():.2f}]")

# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    label = tf.cast(label, tf.int32)
    return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.20)
    image = tf.image.random_contrast(image, 0.80, 1.20)
    image = tf.pad(image, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    image = tf.image.random_flip_left_right(image)   # helps minority letter classes
    image = image + tf.random.normal(tf.shape(image), stddev=0.02)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

train_raw = train_raw.shuffle(20_000, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BS).prefetch(AUTOTUNE)
test_ds_oh = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x): return tf.nn.gelu(x)

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, r]); x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch)
    r2  = residual_block(r1, out_ch)
    r3  = residual_block(r2, out_ch)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    return out

def adaptive_filter_capsule(x, num_classes, capsule_dim=32):
    """
    Lightweight capsule routing.
    capsule_dim=32 — 62-class case-sensitive task; separate A/a routing
    is the core challenge so we keep capsule space rich.
    """
    h = layers.Dense(256, activation=gelu)(x)
    h = layers.Dense(num_classes * capsule_dim)(h)
    h = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps = layers.Multiply()([x_sliced, h])
    caps = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    return layers.BatchNormalization()(caps)

# ─────────────────────────────────────────────────────────────────────────────
#  MODEL  —  4-stage encoder for 62-class depth requirement
# ─────────────────────────────────────────────────────────────────────────────
def build_model(num_classes=62, image_size=28):
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem); stem = layers.Activation(gelu)(stem)

    # ── Encoder: 4 stages for 62-class separation ─────────────────────────
    enc1 = dense_res_block(stem, 64,  64)
    enc2 = dense_res_block(enc1, 64,  128)
    enc3 = dense_res_block(enc2, 128, 256)
    enc4 = dense_res_block(enc3, 256, 512)

    # ── Multi-scale GAP fusion (all 4 stages) ─────────────────────────────
    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc3))
    gap4 = layers.GlobalAveragePooling2D()(enc4)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3, gap4])  # (B, 896)

    # ── Adaptive Filter Capsule ───────────────────────────────────────────
    afc_out = adaptive_filter_capsule(fused_gap, K, capsule_dim=32)   # (B, K)

    # ── Dense head ────────────────────────────────────────────────────────
    x = layers.Dense(256, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation(gelu, name="head_act")(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(K, name="head_logits")(x)

    # ── Gated fusion ──────────────────────────────────────────────────────
    combined   = layers.Concatenate(name="gate_input")([x, afc_out])
    gate       = layers.Dense(2, activation="softmax", name="gate")(combined)
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x, gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc")([afc_out, gate])
    out = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=out, name="WhatNet_ByClass")

# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base         = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────────────────────
model        = build_model(NUM_CLASSES, IMG)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    class_weight=class_weight_dict,          # essential for 62-class imbalance
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=12,
                      restore_best_weights=True, verbose=1),
    ],
)

# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_BYCLASS", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)
print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

# Method

In [ ]:
import os, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG  —  EMNIST ByClass: 814,255 chars, 62 UNBALANCED classes
#  label mapping (0-indexed): 0-9 digits, 10-35 A-Z, 36-61 a-z
#  Hardest EMNIST split: uppercase and lowercase are SEPARATE classes,
#  so 'A'≠'a'. Key confusions: c/C, o/O, p/P, u/U, v/V, w/W, x/X.
#  Severe imbalance — macro F1 is the primary metric, not accuracy.
# ─────────────────────────────────────────────────────────────────────────────
DATASET      = "emnist/byclass"
NUM_CLASSES  = 62
IMG          = 28
BS           = 64
EPOCHS       = 60
LR           = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.10          # severe imbalance + case separation → highest smoothing
WARMUP_EP    = 5
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/byclass"
AUTOTUNE     = tf.data.AUTOTUNE

LABELS = (
    [str(d) for d in range(10)]
    + [chr(c) for c in range(65, 91)]   # A-Z
    + [chr(c) for c in range(97, 123)]  # a-z
)  # 62 labels
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
print("[INFO] Loading EMNIST ByClass via tensorflow_datasets ...")
train_raw, info = tfds.load(
    DATASET, split="train", as_supervised=True, with_info=True, shuffle_files=True,
)
test_raw = tfds.load(DATASET, split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

# ─────────────────────────────────────────────────────────────────────────────
#  CLASS WEIGHTS  —  scan labels to counter severe imbalance
# ─────────────────────────────────────────────────────────────────────────────
print("[INFO] Computing class weights from training labels ...")
label_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, label in train_raw.take(n_train):
    label_counts[int(label.numpy())] += 1

total_samples = label_counts.sum()
class_weights = total_samples / (NUM_CLASSES * np.maximum(label_counts, 1))
class_weights = np.clip(class_weights, 0.1, 10.0)
class_weight_dict = {i: float(class_weights[i]) for i in range(NUM_CLASSES)}
print(f"[INFO] Class weight range: [{class_weights.min():.2f}, {class_weights.max():.2f}]")

# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    label = tf.cast(label, tf.int32)
    return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.20)
    image = tf.image.random_contrast(image, 0.80, 1.20)
    image = tf.pad(image, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    image = tf.image.random_flip_left_right(image)   # helps minority letter classes
    image = image + tf.random.normal(tf.shape(image), stddev=0.02)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

train_raw = train_raw.shuffle(20_000, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BS).prefetch(AUTOTUNE)
test_ds_oh = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x): return tf.nn.gelu(x)

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, r]); x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch)
    r2  = residual_block(r1, out_ch)
    r3  = residual_block(r2, out_ch)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    return out

def adaptive_filter_capsule(x, num_classes, capsule_dim=32):
    """
    Lightweight capsule routing.
    capsule_dim=32 — 62-class case-sensitive task; separate A/a routing
    is the core challenge so we keep capsule space rich.
    """
    h = layers.Dense(256, activation=gelu)(x)
    h = layers.Dense(num_classes * capsule_dim)(h)
    h = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps = layers.Multiply()([x_sliced, h])
    caps = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    return layers.BatchNormalization()(caps)

# ─────────────────────────────────────────────────────────────────────────────
#  MODEL  —  4-stage encoder for 62-class depth requirement
# ─────────────────────────────────────────────────────────────────────────────
def build_model(num_classes=62, image_size=28):
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem); stem = layers.Activation(gelu)(stem)

    # ── Encoder: 4 stages for 62-class separation ─────────────────────────
    enc1 = dense_res_block(stem, 64,  64)
    enc2 = dense_res_block(enc1, 64,  128)
    enc3 = dense_res_block(enc2, 128, 256)
    enc4 = dense_res_block(enc3, 256, 512)

    # ── Multi-scale GAP fusion (all 4 stages) ─────────────────────────────
    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc3))
    gap4 = layers.GlobalAveragePooling2D()(enc4)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3, gap4])  # (B, 896)

    # ── Adaptive Filter Capsule ───────────────────────────────────────────
    afc_out = adaptive_filter_capsule(fused_gap, K, capsule_dim=32)   # (B, K)

    # ── Dense head ────────────────────────────────────────────────────────
    x = layers.Dense(256, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation(gelu, name="head_act")(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(K, name="head_logits")(x)

    # ── Gated fusion ──────────────────────────────────────────────────────
    combined   = layers.Concatenate(name="gate_input")([x, afc_out])
    gate       = layers.Dense(2, activation="softmax", name="gate")(combined)
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x, gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc")([afc_out, gate])
    out = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=out, name="WhatNet_ByClass")

# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base         = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────────────────────
model        = build_model(NUM_CLASSES, IMG)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    class_weight=class_weight_dict,          # essential for 62-class imbalance
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=12,
                      restore_best_weights=True, verbose=1),
    ],
)

# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_BYCLASS", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)
print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

# Output

In [ ]:
[INFO] Using device: cuda
[INFO] Loading EMNIST ByClass ...
[INFO] Train: 628,139 | Val: 69,793 | Test: 116,323
[INFO] Computing class weights ...
[INFO] Class weight range: [0.29, 5.90]
[INFO] Steps/epoch: 9814
[INFO] Trainable params: 21,470,180
Epoch   1/100 | train_loss: 2.1367  train_acc: 52.87%  val_loss: 0.9589  val_acc: 74.86%
         → New best val_acc: 74.86%  (saved)
Epoch   2/100 | train_loss: 1.4147  train_acc: 74.39%  val_loss: 0.7538  val_acc: 78.88%
         → New best val_acc: 78.88%  (saved)
Epoch   3/100 | train_loss: 1.3300  train_acc: 76.75%  val_loss: 0.7378  val_acc: 75.54%
Epoch   4/100 | train_loss: 1.2859  train_acc: 77.77%  val_loss: 0.7098  val_acc: 77.40%
Epoch   5/100 | train_loss: 1.2555  train_acc: 78.62%  val_loss: 0.6816  val_acc: 79.61%
         → New best val_acc: 79.61%  (saved)
Epoch   6/100 | train_loss: 1.2296  train_acc: 79.25%  val_loss: 0.7095  val_acc: 77.05%
Epoch   7/100 | train_loss: 1.2060  train_acc: 79.97%  val_loss: 0.6569  val_acc: 79.66%
         → New best val_acc: 79.66%  (saved)
Epoch   8/100 | train_loss: 1.1906  train_acc: 80.40%  val_loss: 0.6565  val_acc: 80.22%
         → New best val_acc: 80.22%  (saved)
Epoch   9/100 | train_loss: 1.1778  train_acc: 80.81%  val_loss: 0.6284  val_acc: 81.76%
         → New best val_acc: 81.76%  (saved)
Epoch  10/100 | train_loss: 1.1662  train_acc: 81.16%  val_loss: 0.6075  val_acc: 82.61%
         → New best val_acc: 82.61%  (saved)
Epoch  11/100 | train_loss: 1.1570  train_acc: 81.37%  val_loss: 0.6073  val_acc: 81.72%
Epoch  12/100 | train_loss: 1.1503  train_acc: 81.60%  val_loss: 0.6168  val_acc: 82.64%
         → New best val_acc: 82.64%  (saved)
Epoch  13/100 | train_loss: 1.1434  train_acc: 81.79%  val_loss: 0.6604  val_acc: 79.38%
Epoch  14/100 | train_loss: 1.1373  train_acc: 81.98%  val_loss: 0.6390  val_acc: 81.41%
Epoch  15/100 | train_loss: 1.1308  train_acc: 82.18%  val_loss: 0.6139  val_acc: 81.93%
Epoch  16/100 | train_loss: 1.1271  train_acc: 82.33%  val_loss: 0.6039  val_acc: 82.57%
Epoch  17/100 | train_loss: 1.1207  train_acc: 82.50%  val_loss: 0.6110  val_acc: 82.21%
Epoch  18/100 | train_loss: 1.1166  train_acc: 82.62%  val_loss: 0.6183  val_acc: 81.83%
Epoch  19/100 | train_loss: 1.1112  train_acc: 82.79%  val_loss: 0.6191  val_acc: 81.46%
Epoch  20/100 | train_loss: 1.1066  train_acc: 82.95%  val_loss: 0.6144  val_acc: 82.63%
Epoch  21/100 | train_loss: 1.1027  train_acc: 83.03%  val_loss: 0.6031  val_acc: 82.46%
Epoch  22/100 | train_loss: 1.0973  train_acc: 83.16%  val_loss: 0.5818  val_acc: 83.77%
         → New best val_acc: 83.77%  (saved)
Epoch  23/100 | train_loss: 1.0928  train_acc: 83.30%  val_loss: 0.5827  val_acc: 84.68%
         → New best val_acc: 84.68%  (saved)
Epoch  24/100 | train_loss: 1.0894  train_acc: 83.51%  val_loss: 0.5959  val_acc: 82.24%
Epoch  25/100 | train_loss: 1.0854  train_acc: 83.59%  val_loss: 0.5930  val_acc: 83.00%
Epoch  26/100 | train_loss: 1.0804  train_acc: 83.65%  val_loss: 0.5811  val_acc: 83.40%
Epoch  27/100 | train_loss: 1.0752  train_acc: 83.76%  val_loss: 0.5960  val_acc: 83.26%
Epoch  28/100 | train_loss: 1.0702  train_acc: 83.97%  val_loss: 0.5900  val_acc: 82.61%
Epoch  29/100 | train_loss: 1.0668  train_acc: 84.07%  val_loss: 0.5838  val_acc: 83.40%
Epoch  30/100 | train_loss: 1.0608  train_acc: 84.29%  val_loss: 0.5923  val_acc: 82.94%
Epoch  31/100 | train_loss: 1.0549  train_acc: 84.32%  val_loss: 0.5783  val_acc: 83.33%
Epoch  32/100 | train_loss: 1.0508  train_acc: 84.55%  val_loss: 0.5909  val_acc: 83.08%
Epoch  33/100 | train_loss: 1.0457  train_acc: 84.66%  val_loss: 0.5853  val_acc: 83.35%
Epoch  34/100 | train_loss: 1.0403  train_acc: 84.83%  val_loss: 0.5988  val_acc: 82.22%
Epoch  35/100 | train_loss: 1.0347  train_acc: 84.98%  val_loss: 0.5911  val_acc: 83.07%
[INFO] Early stopping at epoch 35 (patience=12)

[RESULT] Test Acc : 84.45%
[RESULT] Macro F1 : 76.20%
[RESULT] Test Loss: 0.5832
[RESULT] Params   : 21,470,180

[RESULT] Worst-5 classes:
         'o' (cls 50) = 15.9%
         'l' (cls 47) = 38.8%
         's' (cls 54) = 39.6%
         'c' (cls 38) = 54.2%
         'u' (cls 56) = 55.6%

[INFO] Saved to ./results/byclass/
[DONE]